# EcoSort AI: build the dataset and train the detector

Run the cells from top to bottom in Google Colab.

Before you start: **Runtime → Change runtime type → T4 GPU**.

## 1. Get the code and libraries

In [ ]:
# Safe to run more than once: it reuses the copy that is already here.
%cd /content
![ -d ecosort-ai ] || git clone https://github.com/TheAnakin01/ecosort-ai
%cd /content/ecosort-ai/ai-model
!git pull
!pip install -q -r requirements.txt

## 2. Download the four Roboflow datasets

Run the cell and paste your Roboflow API key when it asks (Roboflow → Settings → API Keys).
The key is not saved in the notebook.

The cell downloads the newest version of each dataset:

| Folder | Dataset |
|---|---|
| `trash-detection` | [Trash Detection](https://universe.roboflow.com/amongus-o6ilp/trash-detection-1fjjc-ipij8) |
| `trashnet` | [TrashNet: annotated images of trash](https://universe.roboflow.com/polygence-project/trashnet-a-set-of-annotated-images-of-trash-that-can-be-used-for-object-detection) |
| `neat-now` | [Neat Now](https://universe.roboflow.com/fyp-gojku/neat-now) |
| `garbage-cls-3` | [GARBAGE CLASSIFICATION 3](https://universe.roboflow.com/ginas-workspace/garbage-classification-3-p5ly0) |

In [ ]:
from getpass import getpass

from roboflow import Roboflow

# (folder name, workspace, project)
DATASETS = [
    ("trash-detection", "amongus-o6ilp", "trash-detection-1fjjc-ipij8"),
    ("trashnet", "polygence-project", "trashnet-a-set-of-annotated-images-of-trash-that-can-be-used-for-object-detection"),
    ("neat-now", "fyp-gojku", "neat-now"),
    ("garbage-cls-3", "ginas-workspace", "garbage-classification-3-p5ly0"),
]

# .strip() removes spaces or line breaks that a phone can add when pasting.
api_key = getpass("Paste your Roboflow Private API Key: ").strip()

# Describe what arrived without showing the key itself.
print(f"Key received: {len(api_key)} characters")
if not api_key:
    raise SystemExit("Nothing was pasted. Run this cell again, tap the box, paste the key, then press Enter.")
if api_key.startswith("rf_"):
    raise SystemExit(
        "That is the Publishable key (it starts with rf_). Copy the Private API Key "
        "from the same page and run this cell again."
    )
if not api_key.isalnum():
    print("Warning: the key contains spaces or symbols. A Private API Key is only letters and numbers.")

try:
    rf = Roboflow(api_key=api_key)
except RuntimeError as error:
    raise SystemExit(
        f"Roboflow rejected the key. Its reply: {error}\n"
        "Copy the Private API Key from app.roboflow.com -> Settings -> API Keys, "
        "then run this cell again."
    ) from None

for folder, workspace, project_name in DATASETS:
    project = rf.workspace(workspace).project(project_name)
    # A version id looks like "workspace/project/3"; take the highest number.
    latest = max(project.versions(), key=lambda v: int(str(v.id).rsplit("/", 1)[-1]))
    print(f"{folder}: downloading version {latest.id}")
    latest.download("yolov8", location=f"downloads/{folder}")

## 3. Check how the classes will be merged

This prints every class in every dataset and which of our four classes it becomes.
Nothing is copied yet. It reads every image to find duplicates, so it can take a few minutes.

Read the output carefully:
- A class marked `DROPPED` is left out. If it should be one of ours, add a line for it in `configs/class_overrides.yaml` and run this cell again.
- Look at **Boxes per class**. If `organic` is marked as too few, the model will be weak on organic waste until you add your own photos.
- **Identical image files removed** shows how much the datasets overlap. Public datasets often contain each other's photos.

In [ ]:
!python scripts/prepare_dataset.py downloads/trash-detection downloads/trashnet downloads/neat-now downloads/garbage-cls-3 --resplit --dry-run

## 4. Build the merged dataset

`--resplit` ignores each dataset's own train/valid/test folders and makes one fresh split.
Copies of the same photo, and pictures that look the same, always go into the same split,
so the model is never tested on a photo it trained on.

In [ ]:
!rm -rf datasets/ecosort
!python scripts/prepare_dataset.py downloads/trash-detection downloads/trashnet downloads/neat-now downloads/garbage-cls-3 --resplit

## 5. Check the dataset

This looks for problems that would spoil training or make the scores misleading: unreadable images,
broken boxes, and the same photo appearing in more than one split.

- **No errors:** carry on to step 6.
- **Any error:** copy the output and ask for help before training.

Save the table it prints. It goes into `docs/dataset-card.md`.

In [ ]:
!python scripts/validate_dataset.py

## 6. Train

This saves the run straight to your Google Drive, in `ecosort/runs/`. The model file is updated
after every epoch, so if Colab disconnects part-way you still have the best model so far.

- It asks for permission to use your Drive. Allow it.
- It refuses to start until step 5 has passed.
- 30 epochs is a first run. Training stops early if the model stops improving.
- Keep this tab open and the screen on.

The last lines print a table for the **validation** split: precision, recall and AP for each class.
Copy it into `docs/training-log.md`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!python scripts/train.py --epochs 30 --name ecosort-baseline --project /content/drive/MyDrive/ecosort/runs

## 7. Save the dataset records next to the model

These two files say exactly which data the model was trained on.

In [ ]:
import shutil
from pathlib import Path

RUNS = Path("/content/drive/MyDrive/ecosort/runs")
run = max(RUNS.glob("ecosort-baseline*"), key=lambda p: p.stat().st_mtime)
for name in ("dataset_report.json", "validation_report.json"):
    shutil.copy2(Path("datasets/ecosort") / name, run / name)
print("Run folder on Drive:", run)
print(sorted(p.name for p in (run / "weights").iterdir()))

## 8. Look at the results

In [ ]:
from pathlib import Path

from IPython.display import Image, display

RUNS = Path("/content/drive/MyDrive/ecosort/runs")
run = max(RUNS.glob("ecosort-baseline*"), key=lambda p: p.stat().st_mtime)
for name in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    if (run / name).exists():
        display(Image(str(run / name), width=900))

## 9. Download the trained model

The model is already on your Drive. This also downloads it to your device.
Put `best.pt` in `ai-model/weights/` on the machine that runs the backend.

Do not run a test-split evaluation yet. That is the final score, and it is run once,
after the confidence threshold has been chosen on the validation split.

In [ ]:
from pathlib import Path

from google.colab import files

RUNS = Path("/content/drive/MyDrive/ecosort/runs")
run = max(RUNS.glob("ecosort-baseline*"), key=lambda p: p.stat().st_mtime)
files.download(str(run / "weights" / "best.pt"))